# 📐 03. Rule-Based Baseline Accident Detector (MVP)

**Tuần 3 (15/09 – 21/09)** — Kaggle / Colab / Local Notebook

**Mục tiêu:** Xây dựng Baseline MVP phát hiện tai nạn giao thông dựa trên các quy luật động học (kinematic rules) kết hợp với dữ liệu tracking & trajectory trích xuất từ **Tuần 2** (`02_detection_tracking.ipynb`).

## 🔄 Luồng dữ liệu Tuần 2 → Tuần 3
```
Dữ liệu Tuần 2 (trajectories.json / tracking_manifest.json)
         │
         ▼
[Data Loader] — Đọc trajectory, phục hồi lịch sử vị trí & bounding box theo frame
         │
         ▼
[MotionFeatureExtractor] — Vận tốc, Gia tốc, Góc lái, IoU, Hội tụ khoảng cách
         │
         ▼
[RuleBasedAccidentDetector] — Chấm điểm 4 quy luật va chạm (Decel, Turn, IoU, Convergence)
         │
         ▼
[PostProcessor] — Lọc nhiễu thời gian (≥3 frame liên tiếp), Temporal NMS & Cooldown
         │
         ▼
🚨 Danh sách cảnh báo tai nạn + Biểu đồ động học + Video Demo có viền đỏ cảnh báo
```

---

| Cell | Nội dung | Cần GPU? |
|------|----------|----------|
| 1    | Setup môi trường, cấu hình thư mục & imports | Không |
| 2    | Khởi tạo `MotionFeatureExtractor`, `RuleBasedAccidentDetector`, `PostProcessor` | Không |
| 3    | Tìm nạp dữ liệu từ Tuần 2 (`trajectories.json` hoặc `tracking_manifest.json`) | Không |
| 4    | Data Loader: Parser dữ liệu trajectory theo frame & track ID | Không |
| 5    | Chạy Rule-Based Detector trên 1 video mẫu & hiển thị cảnh báo | Không |
| 6    | 📈 Phân tích đồ thị động học (Kinematic Profiles) của vụ va chạm | Không |
| 7    | 🎬 Render Video Demo với Bounding Box cảnh báo tai nạn (tùy chọn) | Không |
| 8    | 🚀 Batch Evaluation trên toàn bộ dataset video Tuần 2 | Không |
| 9    | 📊 Xuất kết quả tổng hợp (`baseline_alerts.json`, `baseline_summary.csv`) | Không |


## ⚙️ Cell 1 — Setup môi trường, cấu hình thư mục & Imports

In [ ]:
import os
import sys
import subprocess
from pathlib import Path
import json, time
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── Phát hiện môi trường ──────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB  = 'google.colab' in sys.modules
print(f'Môi trường thực thi: {"Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "Local"}')

# ── Định vị Repository ───────────────────────────────────────────────────
if IS_KAGGLE:
    WORKSPACE = Path("/kaggle/working")
    REPO_DIR = WORKSPACE / "Traffic-Accident-Detection"
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/ThanhND2005/Traffic-Accident-Detection.git", str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    WORK_DIR = Path('/kaggle/working/week3_output')
    WEEK2_DIR = Path('/kaggle/working/week2_output')
    INPUT_DIR = Path('/kaggle/input')
elif IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE = Path('/content/drive/MyDrive/accident_detection')
    REPO_DIR = DRIVE_BASE / "Traffic-Accident-Detection"
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    WORK_DIR = DRIVE_BASE / 'results/week3_output'
    WEEK2_DIR = DRIVE_BASE / 'results/week2_output'
    INPUT_DIR = DRIVE_BASE / 'datasets'
else:
    # Môi trường Local
    REPO_DIR = Path.cwd()
    if not (REPO_DIR / "src").exists() and (REPO_DIR.parent / "src").exists():
        REPO_DIR = REPO_DIR.parent
        os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    WORK_DIR = REPO_DIR / 'results/week3_output'
    WEEK2_DIR = REPO_DIR / 'results/week2_output'
    INPUT_DIR = REPO_DIR / 'datasets'

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f'✅ Thư mục dự án: {REPO_DIR}')
print(f'✅ Thư mục xuất kết quả Tuần 3: {WORK_DIR}')
print(f'✅ Thư mục dữ liệu Tuần 2: {WEEK2_DIR}')
print(f'✅ Thư mục dataset video: {INPUT_DIR}')

## ⚙️ Cell 2 — Khởi tạo Feature Extractor, Rule Detector & PostProcessor

In [ ]:
from src.features.motion_features import MotionFeatureExtractor
from src.classifiers.rule_based import RuleBasedAccidentDetector
from src.postprocessing.post_processor import PostProcessor

# 1. Bộ trích xuất đặc trưng chuyển động (làm mượt window=3 để giữ nhạy tín hiệu va chạm)
extractor = MotionFeatureExtractor(smoothing_window=3)

# 2. Bộ phân loại theo luật nâng cấp (Enhanced Rule-Based Accident Detector)
# Sử dụng try/except để tương thích 100% với cả môi trường Local lẫn Kaggle/Colab chưa git pull
try:
    rule_detector = RuleBasedAccidentDetector(
        sudden_decel_thresh=4.0,
        direction_change_thresh=40.0,
        iou_overlap_thresh=0.12,
        distance_decrease_rate=0.40,
        min_track_length=4,
        alert_threshold=0.50,
        min_speed_for_turn=3.0,
        decel_drop_ratio=0.50,
    )
except TypeError:
    # Tự động gán thuộc tính nếu chạy trên môi trường Kaggle/Colab chưa pull code mới
    rule_detector = RuleBasedAccidentDetector(
        sudden_decel_thresh=4.0,
        direction_change_thresh=40.0,
        iou_overlap_thresh=0.12,
        distance_decrease_rate=0.40,
        min_track_length=4,
        alert_threshold=0.50,
    )
    rule_detector.min_speed_for_turn = 3.0
    rule_detector.decel_drop_ratio = 0.50

# 3. Bộ hậu xử lý thời gian nâng cao hỗ trợ gom tai nạn 3+ xe (Multi-Vehicle Post-Processor)
class EnhancedPostProcessor(PostProcessor):
    """Hậu xử lý thời gian với khả năng mở rộng chuỗi va chạm đa xe (3+ xe đâm liên hoàn)."""
    def process_frame_candidate(self, frame_id, candidate_event):
        if candidate_event is not None and candidate_event.get("score", 0.0) >= self.warning_thresh:
            self.history_buffer.append((frame_id, candidate_event["score"], candidate_event))
        else:
            self.history_buffer.append((frame_id, 0.0, None))

        # Nếu đang có tai nạn được xác nhận trước đó, kiểm tra xem ứng viên mới có mở rộng vụ tai nạn không
        if candidate_event is not None and candidate_event.get("score", 0.0) >= self.warning_thresh and self.confirmed_events:
            last_event = self.confirmed_events[-1]
            time_gap = frame_id - last_event["frame_end"]

            if time_gap <= max(self.nms_frames, int(self.fps * 2.5)):
                c_tracks = set(candidate_event.get("tracks", []))
                last_tracks = set(last_event.get("tracks", []))
                shares_track = bool(c_tracks & last_tracks)

                from src.features.motion_features import compute_iou
                spatial_overlap = False
                if candidate_event.get("bbox") and last_event.get("bbox"):
                    spatial_overlap = compute_iou(np.array(candidate_event["bbox"]), np.array(last_event["bbox"])) > 0.05

                # Gom các xe đâm bồi vào cùng một vụ tai nạn nếu có chung xe hoặc cùng vùng va chạm
                if shares_track or spatial_overlap:
                    merged_tracks = sorted(list(last_tracks | c_tracks))
                    last_event["tracks"] = merged_tracks
                    last_event["num_vehicles"] = len(merged_tracks)
                    last_event["frame_end"] = frame_id
                    last_event["score"] = max(last_event["score"], candidate_event.get("score", 0.0))
                    cb = candidate_event.get("bbox")
                    lb = last_event.get("bbox")
                    if cb and lb:
                        last_event["bbox"] = [
                            float(min(lb[0], cb[0])),
                            float(min(lb[1], cb[1])),
                            float(max(lb[2], cb[2])),
                            float(max(lb[3], cb[3])),
                        ]
                    self.last_alert_frame = frame_id
                    return None

        if (frame_id - self.last_alert_frame) < self.cooldown_frames:
            return None

        valid_frames = [item for item in self.history_buffer if item[1] >= self.warning_thresh]
        if len(valid_frames) >= self.min_consecutive_alerts:
            best_frame, best_score, best_raw = max(valid_frames, key=lambda x: x[1])

            all_window_tracks = set()
            all_window_boxes = []
            all_window_reasons = []
            for item in valid_frames:
                raw_e = item[2]
                if raw_e:
                    all_window_tracks.update(raw_e.get("tracks", []))
                    if raw_e.get("bbox"):
                        all_window_boxes.append(raw_e["bbox"])
                    if raw_e.get("reasons"):
                        all_window_reasons.append(raw_e["reasons"])

            merged_tracks = sorted(list(all_window_tracks)) if all_window_tracks else best_raw.get("tracks", [])
            num_vehicles = len(merged_tracks)

            if all_window_boxes:
                union_box = [
                    float(min(b[0] for b in all_window_boxes)),
                    float(min(b[1] for b in all_window_boxes)),
                    float(max(b[2] for b in all_window_boxes)),
                    float(max(b[3] for b in all_window_boxes)),
                ]
            else:
                union_box = best_raw.get("bbox", [])

            level = "CRITICAL" if (best_score >= self.high_thresh or num_vehicles >= 3) else "WARNING"

            self.event_counter += 1
            timestamp = frame_id / max(self.fps, 1.0)
            unique_reasons = " | ".join(list(dict.fromkeys(all_window_reasons))) if all_window_reasons else best_raw.get("reasons", "kinematic_anomaly")

            confirmed = {
                "event_id": self.event_counter,
                "frame_start": valid_frames[0][0],
                "frame_end": frame_id,
                "timestamp_sec": round(timestamp, 2),
                "timestamp_str": f"{int(timestamp // 60):02d}:{int(timestamp % 60):02d}.{int((timestamp % 1) * 10):01d}",
                "level": level,
                "score": round(best_score, 3),
                "tracks": merged_tracks,
                "num_vehicles": num_vehicles,
                "bbox": union_box,
                "reasons": unique_reasons,
            }

            self.confirmed_events.append(confirmed)
            self.last_alert_frame = frame_id
            self.history_buffer.clear()
            return confirmed

        return None

post_processor = EnhancedPostProcessor(
    temporal_window=6,
    min_consecutive_alerts=3,
    nms_temporal_seconds=2.0,
    cooldown_seconds=3.0,
    high_confidence_thresh=0.80,
    warning_thresh=0.50,
    fps=30.0,
)

print('✅ Các module Tuần 3 (Hỗ trợ va chạm đa xe 3+ xe) đã được khởi tạo thành công!')
print(f'   - Extractor: smoothing_window={extractor.smoothing_window}')
print(f'   - RuleDetector: decel={rule_detector.sudden_decel_thresh} px/f, turn={rule_detector.direction_change_thresh}°, iou={rule_detector.iou_overlap_thresh}, min_track_len={rule_detector.min_track_length}')
print(f'   - PostProcessor: EnhancedPostProcessor (Hỗ trợ va chạm dây chuyền 3+ xe)')


## 📂 Cell 3 — Tự động tìm nạp dữ liệu từ Tuần 2

Tìm kiếm file kết quả từ Tuần 2 theo thứ tự ưu tiên:
1. `tracking_manifest.json` (kết quả batch dataset nhiều video)
2. Thư mục `trajectories/*.json`
3. `trajectories.json` (kết quả 1 video mẫu)

In [ ]:
# Tìm kiếm file kết quả của Tuần 2 tại các vị trí khả dĩ
search_dirs = [
    WEEK2_DIR,
    REPO_DIR / 'results/week2_output',
    Path('/kaggle/working/week2_output'),
    Path('/kaggle/input/week2-output'),
]

manifest_path = None
manifest_records = []
traj_json_files = []

for sdir in search_dirs:
    if sdir.exists():
        m_cand = sdir / 'tracking_manifest.json'
        if m_cand.exists() and manifest_path is None:
            manifest_path = m_cand
            try:
                with open(manifest_path, 'r', encoding='utf-8') as mf:
                    manifest_records = json.load(mf)
            except Exception as e:
                print(f'⚠️ Lỗi đọc manifest: {e}')
        
        # Tìm các file trajectories.json đơn lẻ hoặc trong thư mục trajectories
        jsons = list(sdir.rglob('*trajectories*.json'))
        for j in jsons:
            if j not in traj_json_files and j.name != 'tracking_manifest.json':
                traj_json_files.append(j)

traj_json_files = sorted(traj_json_files)

print('🔍 KẾT QUẢ TÌM KIẾM DỮ LIỆU TUẦN 2:')
if manifest_path:
    print(f'✅ Tìm thấy Tracking Manifest: {manifest_path} ({len(manifest_records)} video records)')
else:
    print('ℹ️ Không tìm thấy tracking_manifest.json (chạy chế độ danh sách file đơn lẻ)')

print(f'✅ Tìm thấy tổng cộng: {len(traj_json_files)} file trajectory JSON')
for idx, f in enumerate(traj_json_files[:5]):
    size_kb = f.stat().st_size / 1024
    print(f'   [{idx+1}] {f.name} ({size_kb:.1f} KB) -> {f}')
if len(traj_json_files) > 5:
    print(f'   ... và {len(traj_json_files) - 5} file khác.')

if not traj_json_files:
    print('\n⚠️ CẢNH BÁO: Chưa tìm thấy file trajectory nào từ Tuần 2!')
    print('   -> Hãy chắc chắn bạn đã chạy xong 02_detection_tracking.ipynb và xuất kết quả vào results/week2_output/')

## 📥 Cell 4 — Data Loader: Parser dữ liệu Trajectory từ JSON

In [ ]:
def load_trajectory_data(json_path: Path or str):
    """
    Đọc file JSON trajectory từ Tuần 2 và tổ chức lại theo:
    - meta: Thông tin metadata của video
    - tracks: Dict[track_id -> List[state]]
    - frame_lookup: Dict[frame_id -> List[state]] (tiện cho việc quét theo frame)
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    meta = data.get('meta', {})
    raw_tracks = data.get('tracks', {})
    
    tracks = {}
    frame_lookup = defaultdict(list)
    
    for tid_str, states in raw_tracks.items():
        tid = int(tid_str)
        tracks[tid] = states
        for s in states:
            item = dict(s)
            item['track_id'] = tid
            frame_lookup[s['frame_id']].append(item)
            
    return meta, tracks, dict(frame_lookup)

# Kiểm tra nạp thử nghiệm 1 file đầu tiên
if traj_json_files:
    sample_file = traj_json_files[0]
    sample_meta, sample_tracks, sample_frames = load_trajectory_data(sample_file)
    print(f'👉 Nạp thử file mẫu: {sample_file.name}')
    print(f'   - Metadata: {sample_meta}')
    print(f'   - Số lượng tracks: {len(sample_tracks)}')
    print(f'   - Số lượng frames: {len(sample_frames)}')
    
    # Thống kê phân bố độ dài track
    track_lengths = [len(s) for s in sample_tracks.values()]
    if track_lengths:
        print(f'   - Độ dài track: min={min(track_lengths)}, max={max(track_lengths)}, avg={np.mean(track_lengths):.1f} frames')
else:
    sample_tracks, sample_frames = {}, {}

## 🚗 Cell 5 — Chạy Rule-Based Detector trên Video Mẫu

Duyệt qua từng frame của video:
1. Xác định các cặp xe đang ở gần nhau (khoảng cách tâm < `distance_thresh` = 150 px).
2. Trích xuất đặc trưng động học qua `extractor.compute_single_track_features`.
3. Chấm điểm vi phạm quy luật qua `rule_detector.evaluate_pairwise`.
4. Kiểm tra xe tự ngã/lật đơn lẻ qua `rule_detector.detect_single`.
5. Đưa qua `PostProcessor` để xác nhận cảnh báo (tránh cảnh báo ảo).

In [ ]:
def cluster_multi_vehicle_candidates(pairwise_candidates, current_items, current_frame):
    """
    Gom các cặp va chạm thành cụm va chạm đa xe (3+ xe tông nhau trong cùng một khu vực).
    Đảm bảo chạy độc lập không phụ thuộc vào phiên bản src trên máy.
    """
    if not pairwise_candidates:
        return []
        
    from collections import defaultdict
    from src.features.motion_features import compute_iou
    
    adj = defaultdict(set)
    pair_map = defaultdict(list)
    all_candidate_tracks = set()
    
    for c in pairwise_candidates:
        c_tr = c.get('tracks', [])
        for t in c_tr:
            all_candidate_tracks.add(t)
            pair_map[t].append(c)
        if len(c_tr) >= 2:
            for i in range(len(c_tr)):
                for j in range(i + 1, len(c_tr)):
                    adj[c_tr[i]].add(c_tr[j])
                    adj[c_tr[j]].add(c_tr[i])
                    
    visited = set()
    components = []
    for tid in all_candidate_tracks:
        if tid not in visited:
            comp = set()
            q = [tid]
            visited.add(tid)
            while q:
                curr = q.pop(0)
                comp.add(curr)
                for nb in adj[curr]:
                    if nb not in visited:
                        visited.add(nb)
                        q.append(nb)
            components.append(comp)
            
    clustered_results = []
    for comp in components:
        comp_tracks = set(comp)
        comp_reasons = []
        max_score = 0.0
        boxes = []
        
        for tid in comp:
            for c in pair_map[tid]:
                max_score = max(max_score, c['score'])
                comp_reasons.append(c['reasons'])
                boxes.append(c['bbox'])
                
        cluster_bbox = [
            float(min(b[0] for b in boxes)),
            float(min(b[1] for b in boxes)),
            float(max(b[2] for b in boxes)),
            float(max(b[3] for b in boxes)),
        ]
        
        # Kiểm tra xem có xe nào khác trong frame đang chạm/nằm trong vùng va chạm này không
        for it in current_items:
            tid = it['track_id']
            if tid not in comp_tracks:
                b = np.array(it['bbox'], dtype=np.float32)
                cb = np.array(cluster_bbox, dtype=np.float32)
                if compute_iou(b, cb) > 0.08:
                    comp_tracks.add(tid)
                    comp_reasons.append(f'colliding_vehicle(#{tid})')
                    cluster_bbox = [
                        float(min(cluster_bbox[0], b[0])),
                        float(min(cluster_bbox[1], b[1])),
                        float(max(cluster_bbox[2], b[2])),
                        float(max(cluster_bbox[3], b[3])),
                    ]
                    
        num_vehicles = len(comp_tracks)
        final_score = min(1.0, max_score + 0.15) if num_vehicles >= 3 else max_score
        if num_vehicles >= 3:
            comp_reasons.append(f'multi_vehicle_crash({num_vehicles}_vehicles)')
            
        unique_reasons = ' | '.join(list(dict.fromkeys(comp_reasons)))
        clustered_results.append({
            'frame_id': current_frame,
            'tracks': sorted(list(comp_tracks)),
            'num_vehicles': num_vehicles,
            'score': round(float(final_score), 3),
            'rules_triggered': len(comp_reasons),
            'reasons': unique_reasons,
            'bbox': cluster_bbox,
        })
        
    return clustered_results

def analyze_video_trajectories(
    json_path: Path or str,
    detector: RuleBasedAccidentDetector,
    extractor: MotionFeatureExtractor,
    post_proc: PostProcessor,
    distance_thresh: float = 180.0,
    fps: float = 30.0,
):
    """
    Phân tích toàn bộ trajectory của 1 video và trả về danh sách tai nạn được phát hiện.
    Hỗ trợ gom cụm va chạm đa xe (3+ xe đâm nhau liên hoàn).
    """
    meta, tracks, frame_lookup = load_trajectory_data(json_path)
    
    post_proc.history_buffer.clear()
    post_proc.confirmed_events = []
    post_proc.last_alert_frame = -9999
    post_proc.event_counter = 0
    post_proc.fps = fps
    
    sorted_frames = sorted(frame_lookup.keys())
    all_raw_candidates = []
    track_history_cache = defaultdict(list)
    
    for frame_idx in sorted_frames:
        current_items = frame_lookup[frame_idx]
        
        for item in current_items:
            tid = item['track_id']
            track_history_cache[tid].append(item)
            if len(track_history_cache[tid]) > 60:
                track_history_cache[tid].pop(0)
                
        # 1. Đánh giá tương tác giữa các cặp xe gần nhau
        n_items = len(current_items)
        pairwise_candidates = []
        
        for i in range(n_items):
            for j in range(i + 1, n_items):
                item_a = current_items[i]
                item_b = current_items[j]
                tid_a, tid_b = item_a['track_id'], item_b['track_id']
                
                ca = np.array(item_a['center'], dtype=np.float32)
                cb = np.array(item_b['center'], dtype=np.float32)
                dist = np.linalg.norm(ca - cb)
                
                from src.features.motion_features import compute_iou
                box_iou = compute_iou(np.array(item_a['bbox']), np.array(item_b['bbox']))
                
                if dist <= distance_thresh or box_iou > 0.05:
                    hist_a = track_history_cache[tid_a]
                    hist_b = track_history_cache[tid_b]
                    
                    if len(hist_a) >= detector.min_track_length and len(hist_b) >= detector.min_track_length:
                        centers_a = np.array([h['center'] for h in hist_a], dtype=np.float32)
                        bboxes_a  = np.array([h['bbox'] for h in hist_a], dtype=np.float32)
                        centers_b = np.array([h['center'] for h in hist_b], dtype=np.float32)
                        bboxes_b  = np.array([h['bbox'] for h in hist_b], dtype=np.float32)
                        
                        feat_a = extractor.compute_single_track_features(centers_a, bboxes_a)
                        feat_b = extractor.compute_single_track_features(centers_b, bboxes_b)
                        
                        eval_res = detector.evaluate_pairwise(
                            tid_a, tid_b, feat_a, feat_b, bboxes_a, bboxes_b, frame_idx
                        )
                        if eval_res is not None:
                            pairwise_candidates.append(eval_res)
                            
        # 2. Đánh giá trường hợp xe tự ngã / lật đơn lẻ (solo accident)
        for item in current_items:
            tid = item['track_id']
            hist = track_history_cache[tid]
            b = item['bbox']
            if b[0] > 10 and b[1] > 10 and b[2] < 1270 and b[3] < 710:
                if len(hist) >= detector.min_track_length:
                    centers = np.array([h['center'] for h in hist], dtype=np.float32)
                    bboxes  = np.array([h['bbox'] for h in hist], dtype=np.float32)
                    feat    = extractor.compute_single_track_features(centers, bboxes)
                    solo_res = detector.detect_single(tid, feat, bboxes[-1], frame_idx)
                    if solo_res is not None:
                        pairwise_candidates.append(solo_res)

        # 3. Gom cụm va chạm đa xe (Multi-Vehicle Collision Clustering)
        frame_candidate = None
        if pairwise_candidates:
            # Gọi cluster_multi_vehicle_candidates trực tiếp
            clustered = cluster_multi_vehicle_candidates(pairwise_candidates, current_items, frame_idx)
            all_raw_candidates.extend(clustered)
            if clustered:
                frame_candidate = max(clustered, key=lambda x: (x.get('num_vehicles', 1), x.get('score', 0.0)))
                
        # 4. Hậu xử lý thời gian qua PostProcessor (hỗ trợ mở rộng va chạm liên hoàn)
        post_proc.process_frame_candidate(frame_idx, frame_candidate)
        
    confirmed_alerts = post_proc.get_summary()
    return meta, confirmed_alerts, all_raw_candidates

# Chạy phân tích trên video mẫu
if traj_json_files:
    test_json = traj_json_files[0]
    print(f'🚀 Đang phân tích video: {test_json.name}...')
    t0 = time.time()
    meta_res, alerts_res, raw_res = analyze_video_trajectories(
        test_json, rule_detector, extractor, post_processor
    )
    print(f'⏱️ Hoàn thành sau {time.time() - t0:.2f}s!')
    print(f'📊 Kết quả: {len(raw_res)} frame thô nghi vấn -> {len(alerts_res)} VỤ TAI NẠN ĐƯỢC XÁC NHẬN!\n')
    
    if alerts_res:
        print(f'{"ID":<4}{"Thời gian":<12}{"Frame":<14}{"Số xe":<8}{"Mức độ":<10}{"Điểm":<8}{"Danh sách Tracks":<24}{"Lý do cảnh báo"}')
        print('-' * 115)
        for a in alerts_res:
            fr_range = f"{a['frame_start']}-{a['frame_end']}"
            n_veh = a.get('num_vehicles', len(a.get('tracks', [])))
            tr_str = str(a['tracks'])
            print(f"{a['event_id']:<4}{a['timestamp_str']:<12}{fr_range:<14}{n_veh:<8}{a['level']:<10}{a['score']:<8.2f}{tr_str:<24}{a['reasons'][:55]}")
    else:
        print('ℹ️ Không phát hiện vụ tai nạn nghiêm trọng nào trên video này.')


## 📈 Cell 6 — Phân tích đồ thị động học (Kinematic Profiles) của vụ va chạm

Trực quan hóa diễn biến chuyển động tại thời điểm va chạm:
- Biến thiên Vận tốc (Speed) & Gia tốc giảm đột ngột (Deceleration)
- Độ lệch hướng di chuyển (Delta Heading Angle)
- Khoảng cách tương đối & IoU chồng lấn giữa 2 xe

In [ ]:
def plot_collision_kinematics(json_path: Path or str, target_alert: dict, save_path: Path = None):
    """
    Vẽ đồ thị phân tích động học chi tiết của các xe liên quan đến vụ tai nạn (hỗ trợ cả 2 xe và 3+ xe va chạm).
    """
    meta, tracks, _ = load_trajectory_data(json_path)
    tids = target_alert.get('tracks', [])
    if len(tids) < 1:
        print('⚠️ Không có track nào trong alert.')
        return
        
    avail_tids = [t for t in tids if t in tracks]
    if len(avail_tids) < 1:
        print('⚠️ Không tìm thấy tracks trong dữ liệu.')
        return
        
    f_start = target_alert.get('frame_start', 0)
    f_end = target_alert.get('frame_end', f_start + 60)
    window_start = max(0, f_start - 30)
    window_end = f_end + 30
    
    fig, axs = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
    colors = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd', '#ff7f0e']
    
    # 1. Tốc độ di chuyển (Speed Profile)
    for idx, tid in enumerate(avail_tids[:5]):
        c = colors[idx % len(colors)]
        st = tracks[tid]
        f_ids = [s['frame_id'] for s in st if window_start <= s['frame_id'] <= window_end]
        if len(f_ids) < 2:
            continue
        cents = np.array([s['center'] for s in st if window_start <= s['frame_id'] <= window_end], dtype=np.float32)
        vels = np.zeros_like(cents)
        vels[1:] = cents[1:] - cents[:-1]
        sp = np.linalg.norm(vels, axis=1)
        cls_lbl = st[0].get('class_name', '')
        axs[0].plot(f_ids, sp, color=c, linewidth=2, label=f'Speed Xe #{tid} ({cls_lbl})')
        
    axs[0].set_ylabel('Tốc độ (px/frame)')
    n_v = target_alert.get('num_vehicles', len(avail_tids))
    axs[0].set_title(f'Phân Tích Động Học Va Chạm: Event #{target_alert["event_id"]} ({n_v} xe liên quan: {avail_tids})', fontsize=12, fontweight='bold')
    axs[0].legend(loc='upper right')
    axs[0].grid(True, alpha=0.3)
    
    # 2. Gia tốc & Giảm tốc (Deceleration)
    for idx, tid in enumerate(avail_tids[:5]):
        c = colors[idx % len(colors)]
        st = tracks[tid]
        f_ids = [s['frame_id'] for s in st if window_start <= s['frame_id'] <= window_end]
        if len(f_ids) < 3:
            continue
        cents = np.array([s['center'] for s in st if window_start <= s['frame_id'] <= window_end], dtype=np.float32)
        vels = np.zeros_like(cents)
        vels[1:] = cents[1:] - cents[:-1]
        sp = np.linalg.norm(vels, axis=1)
        accel = np.zeros_like(sp)
        accel[1:] = np.abs(sp[1:] - sp[:-1])
        axs[1].plot(f_ids, accel, color=c, linestyle='--', linewidth=1.8, label=f'Decel Xe #{tid}')
        
    axs[1].axhline(y=rule_detector.sudden_decel_thresh, color='r', linestyle=':', label=f'Ngưỡng Decel ({rule_detector.sudden_decel_thresh:.1f} px/f)')
    axs[1].set_ylabel('Độ giảm tốc (px/f²)')
    axs[1].legend(loc='upper right')
    axs[1].grid(True, alpha=0.3)
    
    # 3. Khoảng cách & Vùng va chạm
    if len(avail_tids) >= 2:
        tid_a, tid_b = avail_tids[0], avail_tids[1]
        dict_a = {s['frame_id']: s for s in tracks[tid_a]}
        dict_b = {s['frame_id']: s for s in tracks[tid_b]}
        common_f = sorted(list(set(dict_a.keys()) & set(dict_b.keys())))
        cf_window = [f for f in common_f if window_start <= f <= window_end]
        if cf_window:
            ca = np.array([dict_a[f]['center'] for f in cf_window], dtype=np.float32)
            cb = np.array([dict_b[f]['center'] for f in cf_window], dtype=np.float32)
            dists = np.linalg.norm(ca - cb, axis=1)
            axs[2].plot(cf_window, dists, color='purple', linewidth=2, label=f'Khoảng cách tâm (#{tid_a} vs #{tid_b})')
            
            from src.features.motion_features import compute_iou
            ax3_twin = axs[2].twinx()
            ious = [compute_iou(dict_a[f]['bbox'], dict_b[f]['bbox']) for f in cf_window]
            ax3_twin.plot(cf_window, ious, color='red', linewidth=2, label='IoU Chồng lấn')
            ax3_twin.axhline(y=rule_detector.iou_overlap_thresh, color='red', linestyle=':', label=f'Ngưỡng IoU ({rule_detector.iou_overlap_thresh})')
            ax3_twin.set_ylabel('IoU [0-1]', color='red')
            
    axs[2].set_xlabel('Frame Index')
    axs[2].set_ylabel('Khoảng cách (pixel)', color='purple')
    axs[2].legend(loc='upper left')
    axs[2].grid(True, alpha=0.3)
    
    # Tô màu khoảng thời gian kích hoạt alert
    for ax in axs:
        ax.axvspan(f_start, f_end, color='red', alpha=0.18, label='Vùng Alert Xác Nhận')
        
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f'✅ Đã lưu đồ thị phân tích: {save_path}')
    plt.show()

# Vẽ đồ thị cho vụ tai nạn đầu tiên (nếu có)
if traj_json_files and alerts_res:
    chart_file = WORK_DIR / f'kinematic_{traj_json_files[0].stem}.png'
    plot_collision_kinematics(traj_json_files[0], alerts_res[0], save_path=chart_file)
elif traj_json_files and raw_res:
    best_raw = max(raw_res, key=lambda x: x['score'])
    chart_file = WORK_DIR / f'kinematic_raw_{traj_json_files[0].stem}.png'
    plot_collision_kinematics(traj_json_files[0], {'tracks': best_raw['tracks'], 'event_id': 0, 'frame_start': best_raw['frame_id']-5, 'frame_end': best_raw['frame_id']}, save_path=chart_file)


## 🎬 Cell 7 — (Tùy chọn) Render Video Demo với Cảnh báo Bounding Box Đỏ

Nếu có file video gốc (`.mp4`), cell này sẽ render video output với:
- Bounding box xanh cho các xe bình thường
- Đuôi trajectory di chuyển
- **Bounding box ĐỎ** + Banner cảnh báo khi phát hiện va chạm

In [ ]:
import cv2

def is_valid_video(vid_p: Path or str) -> bool:
    """Kiểm tra xem file video có đọc được không (tránh lỗi moov atom not found do file rỗng/hỏng)."""
    if not vid_p:
        return False
    vp = Path(vid_p)
    if not vp.exists() or vp.stat().st_size < 1024:
        return False
    test_cap = cv2.VideoCapture(str(vp))
    if not test_cap.isOpened():
        test_cap.release()
        return False
    w = int(test_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(test_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cnt = int(test_cap.get(cv2.CAP_PROP_FRAME_COUNT))
    test_cap.release()
    return w > 0 and h > 0 and cnt > 0

def render_alert_video(
    video_path: Path or str,
    json_path: Path or str,
    alerts: list,
    output_path: Path or str,
):
    """
    Render video kết quả với annotation cảnh báo tai nạn.
    Tất cả các xe trong vụ va chạm (kể cả vụ va chạm 3+ xe) đều được đánh dấu hộp ĐỎ nổi bật.
    """
    if not os.path.exists(video_path):
        print(f'⚠️ Video không tồn tại: {video_path}')
        return
        
    meta, tracks, frame_lookup = load_trajectory_data(json_path)
    
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f'❌ Không thể mở video (file bị lỗi hoặc thiếu moov atom do ghi chưa xong): {video_path}')
        return
        
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    if W == 0 or H == 0 or total_frames <= 0:
        print(f'❌ Video không hợp lệ (W={W}, H={H}, frames={total_frames}): {video_path}')
        cap.release()
        return
    
    out_writer = cv2.VideoWriter(
        str(output_path),
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps,
        (W, H)
    )
    
    print(f'🎬 Bắt đầu render video: {output_path} ({total_frames} frames)...')
    t0 = time.time()
    
    # Tạo lookup nhanh cho các frame có alert (mở rộng thêm 90 frame = 3 giây sau va chạm để người xem quan sát)
    alert_banner_map = {}
    for a in alerts:
        f_start = a['frame_start']
        f_end = min(total_frames, a['frame_end'] + int(fps * 3.5))
        for f in range(f_start, f_end):
            alert_banner_map[f] = a
            
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        current_items = frame_lookup.get(frame_idx, [])
        has_alert = frame_idx in alert_banner_map
        cur_alert = alert_banner_map.get(frame_idx)
        alert_tracks = cur_alert.get('tracks', []) if has_alert else []
        alert_track_set = set(alert_tracks)
        
        # 1. Vẽ bounding box các xe trong frame
        for item in current_items:
            tid = item['track_id']
            b = [int(v) for v in item['bbox']]
            is_collision_vehicle = (tid in alert_track_set)
            
            if is_collision_vehicle:
                # Xe liên quan đến vụ tai nạn: viền đỏ đậm + nhãn cảnh báo đỏ
                color = (0, 0, 255)
                thick = 3
                cv2.rectangle(frame, (b[0], b[1]), (b[2], b[3]), color, thick)
                label = f"🚨 CRASH #{tid} {item.get('class_name', '')}"
                # Nền chữ đỏ cho nhãn xe tai nạn
                (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
                cv2.rectangle(frame, (b[0], max(0, b[1] - th - 8)), (b[0] + tw + 4, b[1]), (0, 0, 200), -1)
                cv2.putText(frame, label, (b[0] + 2, max(12, b[1] - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)
            else:
                # Xe bình thường: viền xanh lá
                color = (0, 255, 0)
                thick = 2
                cv2.rectangle(frame, (b[0], b[1]), (b[2], b[3]), color, thick)
                label = f"#{tid} {item.get('class_name', '')}"
                cv2.putText(frame, label, (b[0], max(15, b[1] - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
            
        # 2. Vẽ Banner cảnh báo trên đỉnh màn hình nếu có tai nạn
        if has_alert:
            n_veh = cur_alert.get('num_vehicles', len(alert_tracks))
            # Banner màu đỏ trên cùng
            cv2.rectangle(frame, (0, 0), (W, 65), (0, 0, 180), -1)
            alert_text = f"🚨 ACCIDENT ALERT [{cur_alert['level']}] Score: {cur_alert['score']:.2f} | {n_veh} XE VA CHẠM (Tracks: {alert_tracks})"
            reason_text = f"Lý do: {cur_alert['reasons']}"
            cv2.putText(frame, alert_text, (20, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (255, 255, 255), 2)
            cv2.putText(frame, reason_text, (20, 52), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (200, 255, 255), 1)
            
            # Vẽ khung đỏ bao quanh toàn bộ cụm va chạm (union bbox)
            if 'bbox' in cur_alert and cur_alert['bbox']:
                ub = [int(v) for v in cur_alert['bbox']]
                cv2.rectangle(frame, (ub[0], ub[1]), (ub[2], ub[3]), (0, 0, 255), 2, lineType=cv2.LINE_AA)
        else:
            # Status bar bình thường
            cv2.putText(frame, f"Frame: {frame_idx} | Tracks: {len(current_items)}", (15, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            
        out_writer.write(frame)
        frame_idx += 1
        
    cap.release()
    out_writer.release()
    print(f'✅ Video demo đã lưu thành công: {output_path} ({time.time() - t0:.1f}s)')

# ── TÌM KIẾM VIDEO ĐẦU VÀO ĐỂ RENDER ANNOTATION ────────────────────────────
sample_video_cand = None
is_fallback_viz = False

if traj_json_files:
    target_traj_file = traj_json_files[0]
    target_stem = target_traj_file.stem.replace('_trajectories', '').replace('trajectories', '').strip('_')
    
    # 1. Thử lấy từ metadata của file trajectory nếu có
    if 'sample_meta' in locals() and isinstance(sample_meta, dict):
        cand_p = sample_meta.get('video_path')
        if cand_p and Path(cand_p).exists() and is_valid_video(cand_p):
            sample_video_cand = Path(cand_p)
            print(f'🎯 Tìm thấy video gốc từ metadata JSON: {sample_video_cand.name}')
            
    # 2. Thử tra cứu từ tracking_manifest.json nếu có
    if sample_video_cand is None and 'manifest_records' in locals() and manifest_records:
        for rec in manifest_records:
            if rec.get('video_stem') == target_stem or rec.get('trajectory_json') == str(target_traj_file) or not target_stem:
                cand_p = rec.get('video_path')
                if cand_p and Path(cand_p).exists() and is_valid_video(cand_p):
                    sample_video_cand = Path(cand_p)
                    print(f'🎯 Tìm thấy video gốc từ tracking_manifest.json: {sample_video_cand.name}')
                    break

    # 3. Quét các thư mục tìm kiếm video đa môi trường (Kaggle / Colab / Local)
    if sample_video_cand is None:
        video_dirs_to_check = [
            WEEK2_DIR,
            REPO_DIR / 'demo/sample_videos',
            REPO_DIR / 'datasets/cadp/videos',
            REPO_DIR / 'datasets',
            REPO_DIR / 'data',
        ]
        if IS_KAGGLE:
            video_dirs_to_check.extend([
                Path('/kaggle/input'),
                Path('/kaggle/working/cadp_videos'),
                Path('/kaggle/working'),
            ])
        elif IS_COLAB:
            video_dirs_to_check.extend([
                DRIVE_BASE / 'datasets',
                DRIVE_BASE / 'results/sample_videos',
                DRIVE_BASE / 'datasets/cadp/videos',
            ])

        # Quét ưu tiên theo stem khớp tên trước
        for vdir in video_dirs_to_check:
            if not vdir.exists():
                continue
            all_vids = []
            for ext in ['*.mp4', '*.avi', '*.mkv', '*.mov']:
                try:
                    all_vids.extend(list(vdir.rglob(ext) if vdir != WEEK2_DIR else vdir.glob(ext)))
                except Exception:
                    pass
                
            # Ưu tiên video trùng stem với trajectory
            for cand in all_vids:
                if cand.name.startswith(('demo_alert_', 'trajectory_visualization')):
                    continue
                if target_stem and (cand.stem == target_stem or target_stem in cand.stem):
                    if is_valid_video(cand):
                        sample_video_cand = cand
                        print(f'🎯 Tìm thấy video gốc khớp stem [{target_stem}]: {cand.name}')
                        break
            if sample_video_cand:
                break
                
        # Nếu chưa thấy khớp stem, lấy bất kỳ video gốc nào hợp lệ trong các thư mục
        if sample_video_cand is None:
            for vdir in video_dirs_to_check:
                if not vdir.exists():
                    continue
                for ext in ['*.mp4', '*.avi', '*.mkv', '*.mov']:
                    try:
                        cands = list(vdir.rglob(ext) if vdir != WEEK2_DIR else vdir.glob(ext))
                    except Exception:
                        cands = []
                    for cand in cands:
                        if cand.name.startswith(('demo_alert_', 'trajectory_visualization')):
                            continue
                        if is_valid_video(cand):
                            sample_video_cand = cand
                            print(f'🎯 Tìm thấy video mẫu hợp lệ: {cand.name} (tại {vdir})')
                            break
                    if sample_video_cand:
                        break
                if sample_video_cand:
                    break

    # 4. Fallback: Nếu không tìm thấy video gốc raw, dùng trajectory_visualization.mp4 từ Tuần 2
    if sample_video_cand is None:
        viz_cand = WEEK2_DIR / 'trajectory_visualization.mp4'
        if not viz_cand.exists():
            viz_cands = list(WEEK2_DIR.glob('*trajectory_visualization*.mp4'))
            if viz_cands:
                viz_cand = viz_cands[0]
        if viz_cand.exists() and is_valid_video(viz_cand):
            sample_video_cand = viz_cand
            is_fallback_viz = True
            print(f'ℹ️ Lưu ý: Không tìm thấy file video gốc raw. Tự động sử dụng video visualization từ Tuần 2:')
            print(f'   -> {sample_video_cand.name}')

if sample_video_cand and traj_json_files:
    out_vid_name = f'demo_alert_{sample_video_cand.name}'
    if is_fallback_viz and not out_vid_name.endswith('_with_alerts.mp4'):
        out_vid_name = 'demo_alert_trajectory_visualization.mp4'
    out_vid_file = WORK_DIR / out_vid_name
    render_alert_video(sample_video_cand, traj_json_files[0], alerts_res, out_vid_file)
else:
    print('ℹ️ Bỏ qua render video (không tìm thấy video nào để render).')
    print('   💡 Mẹo: Bạn có thể đặt file video mẫu .mp4 vào:')
    print(f'      1. {WEEK2_DIR}')
    print(f'      2. {REPO_DIR / "demo/sample_videos"}')
    print(f'      3. Hoặc gán trực tiếp: sample_video_cand = Path("duong_dan_video.mp4")')


## 🚀 Cell 8 — Batch Evaluation trên toàn bộ Dataset Tuần 2

Chạy tự động Rule-Based Detector trên toàn bộ các file trajectory thu được từ batch processing của Tuần 2.

In [ ]:
print(f'🚀 Bắt đầu Batch Evaluation trên {len(traj_json_files)} video...')

batch_results = []
total_batch_alerts = []
t_batch_0 = time.time()

for idx, json_p in enumerate(traj_json_files):
    vstem = json_p.stem.replace('_trajectories', '')
    t_start = time.time()
    
    meta_i, alerts_i, raw_i = analyze_video_trajectories(
        json_p, rule_detector, extractor, post_processor
    )
    
    elapsed = time.time() - t_start
    n_alerts = len(alerts_i)
    
    # Gắn thông tin video vào alert
    for a in alerts_i:
        a_record = dict(a)
        a_record['video_stem'] = vstem
        total_batch_alerts.append(a_record)
        
    batch_results.append({
        'video_stem': vstem,
        'trajectory_file': json_p.name,
        'total_tracks': meta_i.get('total_tracks', 0),
        'last_frame': meta_i.get('last_frame', 0),
        'raw_candidates': len(raw_i),
        'confirmed_accidents': n_alerts,
        'elapsed_sec': round(elapsed, 2),
        'status': 'HAS_ACCIDENT' if n_alerts > 0 else 'NORMAL'
    })
    
    status_icon = '🚨' if n_alerts > 0 else '✅'
    print(f'[{idx+1:3d}/{len(traj_json_files)}] {status_icon} {vstem:<30} | {meta_i.get("total_tracks", 0):2d} tracks | {n_alerts} tai nạn | {elapsed:.2f}s')

print('\n' + '='*75)
print(f'🎉 HOÀN THÀNH BATCH EVALUATION {len(batch_results)} VIDEO!')
print(f'⏱️ Tổng thời gian: {time.time() - t_batch_0:.1f}s')
print(f'🚨 Tổng số vụ tai nạn phát hiện: {len(total_batch_alerts)}')
print('='*75)

## 💾 Cell 9 — Xuất kết quả tổng hợp & Báo cáo Deliverables Tuần 3

In [ ]:
# 1. Xuất file tổng hợp các vụ tai nạn phát hiện được
alerts_out_path = WORK_DIR / 'baseline_alerts.json'
with open(alerts_out_path, 'w', encoding='utf-8') as f:
    json.dump(total_batch_alerts, f, indent=2, ensure_ascii=False)

# 2. Xuất bảng thống kê theo từng video
df_summary = pd.DataFrame(batch_results)
summary_csv_path = WORK_DIR / 'baseline_summary.csv'
df_summary.to_csv(summary_csv_path, index=False)

print(f'✅ Đã lưu kết quả chi tiết các vụ tai nạn: {alerts_out_path}')
print(f'✅ Đã lưu bảng tổng hợp đánh giá: {summary_csv_path}\n')

# 3. Hiển thị bảng tổng kết
print('📋 BẢNG TỔNG KẾT BASELINE RULE-BASED DETECTOR:')
print(df_summary.to_string(index=False))

print('\n' + '='*75)
print('🎯 CHECKLIST DELIVERABLES TUẦN 3 (RULE-BASED MVP):')
print(' [x] Nạp thành công dữ liệu tracking & trajectory từ Tuần 2')
print(' [x] Module MotionFeatureExtractor trích xuất đặc trưng động học')
print(' [x] Module RuleBasedAccidentDetector đánh giá 4 quy luật va chạm')
print(' [x] Module PostProcessor lọc nhiễu thời gian & chống spam cảnh báo')
print(' [x] Đồ thị phân tích động học va chạm (Speed, Decel, Angle, IoU)')
print(' [x] Xuất baseline_alerts.json & baseline_summary.csv cho toàn bộ dataset')
print('='*75)